# 윈도우 함수 심화 — 실습 노트북

> 5교시(직접 재봅니다) 실험 1~3, 6교시(필수 실습 — 세션 복원), 7교시(도전 과제 1~3)를 여기서 실제로
> 실행합니다. 오늘 앞 파트 **"PostgreSQL 기초 및 고급 쿼리 작성 실습"** 1교시에서 띄운 `db-pg`
> 컨테이너와 `course_db`를 그대로 씁니다 — 별도 설치나 컨테이너 재기동이 필요 없습니다.

## 실행 안내
- 🐳 **선행 조건**: `db-pg` 컨테이너가 이미 켜져 있어야 합니다(오늘 1교시에서 띄운 것 그대로).
- 🔐 비밀번호는 코드에 하드코딩하지 않고 같은 폴더 `.env` 파일(`PGPASSWORD=postgres`)에서 `load_dotenv()`로 읽습니다.
- 위에서 아래로 **순서대로** 실행하세요. `user_events`는 **`course_db`에 실제로 만드는 영구 테이블**이라
  다음 주 JSONB 시간에도 그대로 씁니다. `score_board`·`daily_errors`는 실습용 임시(TEMP) 테이블이라
  이 노트북 연결이 끊기면 사라집니다(재실행하면 다시 만들어집니다).


In [1]:
# ✅ 포인트: psycopg = Python용 PostgreSQL 드라이버. 오늘 앞 파트(PostgreSQL 기초)와 동일한 버전을 씁니다.
# 🆕 psycopg 'v3'를 씁니다(import psycopg). 과거의 psycopg2와 다릅니다.
#
# 📌 %pip install 이란? 주피터 노트북 안에서 파이썬 패키지를 설치하는 명령입니다.
#    - 맨 앞의 %는 "매직 커맨드"라는 뜻으로, 셸(터미널) 명령이 아니라 주피터 전용 명령임을 표시합니다.
#    - -q 는 "quiet"의 약자로, 설치 로그를 최소한으로만 출력하라는 옵션입니다.
#    - "psycopg[binary]==3.3.4" 는 psycopg 패키지의 정확히 3.3.4 버전을, binary(컴파일 없이 바로
#      쓸 수 있는 형태)로 설치하라는 뜻입니다. ==으로 버전을 고정하면 나중에 다른 버전 때문에
#      코드가 안 되는 문제를 예방할 수 있습니다.
#    - python-dotenv 는 .env 파일에서 환경변수를 읽어오는 패키지입니다(바로 다음 셀에서 사용).
%pip install -q "psycopg[binary]==3.3.4" python-dotenv

# import psycopg: 방금 설치한 패키지를 파이썬 코드에서 쓸 수 있게 불러옵니다.
import psycopg
print("psycopg 버전:", psycopg.__version__)  # 실제로 설치된 버전을 확인 — 3.3.4가 출력되어야 정상입니다.


Note: you may need to restart the kernel to use updated packages.
psycopg 버전: 3.3.4


## 접속 — psycopg & .env

오늘 만든 `user_events`는 `course_db` 안에 만듭니다 — `log_events`·`hourly_error_stats`와 같은
데이터베이스에 나란히 쌓입니다. 비밀번호는 `.env`의 `PGPASSWORD`에서 읽습니다(하드코딩 금지).


In [2]:
import os
from dotenv import load_dotenv

# load_dotenv(): 같은 폴더에 있는 .env 파일을 읽어서, 그 안의 KEY=VALUE 줄들을
# 파이썬의 "환경변수"로 등록해줍니다. 환경변수는 운영체제 차원에서 프로그램이 공유하는
# 설정값 저장소라고 생각하면 됩니다 — 코드에 비밀번호를 직접 적지 않기 위한 표준적인 방법입니다.
load_dotenv()                      # 같은 폴더의 .env → 환경변수

# os.environ["PGPASSWORD"]: 위에서 등록된 환경변수 중 PGPASSWORD라는 이름의 값을 꺼냅니다.
# (.env 파일 안에 PGPASSWORD=postgres 라고 적혀 있으므로 pw에는 "postgres"가 담깁니다.)
pw = os.environ["PGPASSWORD"]      # 비밀번호는 .env 파일에 (하드코딩 금지)

# psycopg.connect(...): PostgreSQL 서버에 실제로 접속합니다. 괄호 안은 접속 정보입니다.
#   - host="localhost": 내 컴퓨터(도커 컨테이너가 떠 있는 곳)에 접속
#   - port=5432: PostgreSQL의 기본 포트 번호
#   - dbname="course_db": 접속할 데이터베이스 이름 (오늘 1교시에 만든 것)
#   - user="postgres", password=pw: 로그인 계정과 비밀번호
conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)

# cur = conn.cursor(): "커서(cursor)"는 SQL 문을 실제로 실행하고 결과를 받아오는 창구입니다.
# 연결(conn) 하나에 여러 개의 커서를 만들 수 있지만, 이 실습에서는 이 cur 하나만 계속 재사용합니다.
cur = conn.cursor()
print("연결 완료 →", conn)


연결 완료 → <psycopg.Connection [IDLE] (host=localhost user=postgres database=course_db) at 0x2a158a56a50>


In [3]:
# 실습 편의용 출력 헬퍼 — 교안의 "예상 결과" 표와 같은 모양으로 찍어줍니다.
# (이 두 함수는 SQL 문법과는 무관한, 순수 파이썬 코드입니다 — 결과를 예쁘게 표 모양으로 보여주기 위한 도구일 뿐입니다.)

def show(rows, cols):
    # rows: SQL 조회 결과 (행들의 리스트, 각 행은 튜플). cols: 컬럼 이름 리스트.
    # 아래 줄: 각 행에서 값이 None(=SQL의 NULL)인 경우 화면에는 빈 문자열("")로 바꿔서 보여줍니다.
    #   "" if v is None else v  → 파이썬의 조건 표현식(삼항 연산자와 비슷): v가 None이면 "", 아니면 v 그대로.
    rows = [tuple("" if v is None else v for v in r) for r in rows]

    # 각 컬럼마다 "가장 긴 값의 글자수"를 구해서 표의 칸 너비(width)로 씁니다 — 줄이 깔끔하게 맞춰지도록.
    #   max(len(str(c)), *(len(str(r[i])) for r in rows)) : 컬럼명 길이와 모든 행의 값 길이 중 최댓값
    #   if rows else len(str(c)) : 행이 하나도 없으면(빈 결과) 컬럼명 길이만 사용
    widths = [max(len(str(c)), *(len(str(r[i])) for r in rows)) if rows else len(str(c))
              for i, c in enumerate(cols)]

    # 헤더(컬럼명) 한 줄 출력. ljust(w)는 문자열을 왼쪽 정렬하고 오른쪽을 공백으로 채워 폭을 w로 맞춥니다.
    print(" | ".join(str(c).ljust(w) for c, w in zip(cols, widths)))
    # 헤더 밑에 "-----+-----" 같은 구분선을 그립니다.
    print("-+-".join("-" * w for w in widths))
    # 실제 데이터 행들을 한 줄씩 출력합니다.
    for r in rows:
        print(" | ".join(str(r[i]).ljust(w) for i, w in enumerate(widths)))

def run(sql, params=None):
    # SELECT 실행 + 결과 반환 (컬럼명 포함)
    # cur.execute(sql, params): SQL 문자열을 실제로 데이터베이스에 보내 실행합니다.
    #   params는 SQL 인젝션을 막기 위한 "안전한 값 전달" 방법이지만, 이 노트북에서는 대부분 사용하지 않습니다(None).
    cur.execute(sql, params)
    # cur.description: 방금 실행한 SELECT 결과의 컬럼 정보(이름, 타입 등)를 담고 있습니다.
    #   d.name으로 각 컬럼의 이름만 꺼내 리스트로 만듭니다.
    cols = [d.name for d in cur.description]
    # cur.fetchall(): 실행 결과의 모든 행을 파이썬 리스트로 가져옵니다.
    # 이 함수는 (행들, 컬럼명들) 두 가지를 함께 돌려줘서, 위의 show() 함수에 바로 넘길 수 있게 합니다.
    return cur.fetchall(), cols


---
## 2교시 · 왜 GROUP BY만으로는 부족한가

`GROUP BY`는 여러 행을 하나의 요약 행으로 압축합니다 — 압축되고 나면 원본 행에는 다시 접근할 수 없습니다. 작은 데모 테이블로 이 한계를 먼저 겪어본 뒤, 윈도우 함수가 어떻게 "행을 유지한 채" 같은 문제를 푸는지 확인합니다. (교안 2교시 모듈 2-1·2-2)

In [4]:
# DROP TABLE IF EXISTS demo_events: demo_events라는 이름의 테이블이 이미 있으면 먼저 지웁니다.
#   (여러 번 반복 실행해도 "이미 테이블이 있습니다" 오류가 나지 않도록 하는 안전장치입니다.)
cur.execute("DROP TABLE IF EXISTS demo_events")

# CREATE TEMP TABLE: "임시 테이블"을 만듭니다. TEMP(TEMPORARY)이므로 이 노트북의 DB 연결이
# 끊기면(커널 재시작 등) 자동으로 사라집니다 — 실습용 데이터라 영구 저장할 필요가 없기 때문입니다.
# 괄호 안은 "컬럼명  타입" 쌍의 목록입니다.
cur.execute('''
    CREATE TEMP TABLE demo_events (
        user_id      TEXT,        -- 사용자 ID (문자열)
        event_time   TIMESTAMP,   -- 이벤트가 발생한 시각 (날짜+시간)
        event_type   TEXT,        -- 이벤트 종류: 'click'(클릭), 'view'(조회), 'buy'(구매) 등
        duration_sec INT          -- 이벤트가 지속된 시간(초), 정수
    )
''')

# INSERT INTO ... VALUES (...), (...), ...: 여러 행을 한 번에 삽입합니다.
# 각 괄호가 한 행이고, 괄호 안 순서는 위 CREATE TABLE에서 선언한 컬럼 순서(user_id, event_time, event_type, duration_sec)와 같아야 합니다.
cur.execute('''
    INSERT INTO demo_events VALUES
        ('A', '2026-09-16 09:01:00', 'click', 5),
        ('A', '2026-09-16 09:15:00', 'view',  40),
        ('A', '2026-09-16 09:45:00', 'buy',   90),
        ('A', '2026-09-16 10:20:00', 'view',  30),
        ('A', '2026-09-16 11:05:00', 'click', 8),
        ('B', '2026-09-16 10:02:00', 'click', 6),
        ('B', '2026-09-16 10:30:00', 'view',  25),
        ('B', '2026-09-16 11:40:00', 'buy',   60)
''')

# conn.commit(): 지금까지의 변경사항(테이블 생성, 데이터 삽입)을 데이터베이스에 "확정"합니다.
# PostgreSQL은 commit을 호출하기 전까지는 변경을 임시 상태로만 두므로, 실습 중 commit을 잊으면
# 다른 셀에서 방금 넣은 데이터가 안 보일 수 있습니다 — 이 노트북에서는 데이터를 바꿀 때마다 습관적으로 호출합니다.
conn.commit()

# 방금 넣은 데이터를 확인해봅니다. ORDER BY user_id, event_time: 사용자별로 묶고, 그 안에서 시간순 정렬.
rows, cols = run("SELECT * FROM demo_events ORDER BY user_id, event_time")
show(rows, cols)

user_id | event_time          | event_type | duration_sec
--------+---------------------+------------+-------------
A       | 2026-09-16 09:01:00 | click      | 5           
A       | 2026-09-16 09:15:00 | view       | 40          
A       | 2026-09-16 09:45:00 | buy        | 90          
A       | 2026-09-16 10:20:00 | view       | 30          
A       | 2026-09-16 11:05:00 | click      | 8           
B       | 2026-09-16 10:02:00 | click      | 6           
B       | 2026-09-16 10:30:00 | view       | 25          
B       | 2026-09-16 11:40:00 | buy        | 60          


### 시도 1 · GROUP BY로 "각 사용자 최근 2건" 뽑아보기

`GROUP BY`는 `MAX(event_time)` 같은 집계값 1개만 사용자당 1행으로 돌려줍니다. "최근 2건"처럼 여러 행을 원본 그대로 뽑는 것은 `GROUP BY`만으로는 불가능합니다.

In [5]:
# GROUP BY는 사용자당 1행으로 압축됩니다 — "최근 2건"을 뽑을 수 없습니다
# GROUP BY user_id: user_id가 같은 행들을 하나의 그룹으로 묶습니다.
# 그룹으로 묶고 나면, SELECT에는 "그룹 전체에 대한 집계값"만 쓸 수 있습니다 — 원본 행 하나하나가 아니라요.
#   - MAX(event_time): 그 그룹(사용자) 안에서 가장 늦은(최신) 시각 1개만 남습니다.
#   - COUNT(*): 그 그룹 안에 행이 몇 개 있었는지 개수만 남습니다.
# 즉 원래 5행이었던 user 'A'의 데이터가 "1행"으로 요약되어 버립니다 — 어떤 이벤트였는지는 사라집니다.
rows, cols = run('''
    SELECT user_id, MAX(event_time) AS latest_time, COUNT(*) AS total_events
    FROM demo_events
    GROUP BY user_id
''')
show(rows, cols)
# ← user_id당 1행뿐. 원본 이벤트(어떤 event_type이었는지 등)는 이미 사라졌습니다.

user_id | latest_time         | total_events
--------+---------------------+-------------
B       | 2026-09-16 11:40:00 | 3           
A       | 2026-09-16 11:05:00 | 5           


### 시도 2 · 윈도우 함수로 다시 (행을 유지한 채)

`ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time DESC)`로 각 행에 "이 사용자 안에서 몇 번째로 최신인지" 번호를 매기면, 원본 행을 그대로 둔 채 순위만 열로 추가할 수 있습니다. 다만 윈도우 함수는 `SELECT` 단계에서 계산되므로 같은 쿼리의 `WHERE`에서 바로 참조할 수 없습니다(교안 4교시 모듈 4-3에서 이유를 다룹니다) — 먼저 오류를 확인하고, CTE로 감싸 해결합니다.

In [6]:
# ❌ 윈도우 함수 결과는 WHERE에서 바로 못 씁니다 (SELECT 단계에서 계산되기 때문 — 교안 4교시 모듈 4-3)
#
# try/except: 파이썬에서 "오류가 날 수도 있는 코드"를 안전하게 실행하는 문법입니다.
#   try 블록 안의 코드를 실행하다가 오류(예외)가 발생하면, 프로그램이 멈추지 않고 except 블록으로 넘어갑니다.
#   여기서는 "일부러" 오류가 나는 SQL을 실행해서, 학생들이 실제 오류 메시지를 직접 눈으로 보게 하는 목적입니다.
try:
    cur.execute('''
        SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time DESC) AS rn
        FROM demo_events
        WHERE rn <= 2
    ''')
    # ↑ rn은 SELECT 절에서 새로 계산되는 값인데, SQL은 WHERE 절을 SELECT보다 먼저 처리하기 때문에
    #   WHERE가 실행되는 시점에는 아직 rn이라는 컬럼이 존재하지 않습니다 → 오류 발생
except Exception as e:
    # PostgreSQL은 오류가 나면 그 연결(트랜잭션)이 "오류 상태"로 잠깁니다.
    # conn.rollback(): 오류가 난 시점 이전 상태로 되돌려서, 다음 SQL을 다시 정상적으로 실행할 수 있게 합니다.
    #   (이걸 안 하면 "current transaction is aborted" 라는 후속 오류가 계속 납니다.)
    conn.rollback()
    print("❌ 예상된 오류:", e)

# ✅ CTE로 감싸면 정상 동작 — 각 사용자 최근 2건이 그대로 나옵니다 (행은 그대로, 열만 추가됨)
#
# WITH ranked AS (...): CTE(Common Table Expression, 공통 테이블 표현식)라고 부릅니다.
#   괄호 안의 SELECT 결과를 "ranked"라는 임시 이름의 표처럼 취급해, 바깥쪽 SELECT에서 재사용할 수 있게 해줍니다.
#   즉 1단계로 rn(순번)까지 계산을 "먼저 끝내고", 2단계에서 그 결과표(ranked)를 대상으로 WHERE를 겁니다.
#   이렇게 단계를 나누면 WHERE가 이미 계산되어 있는 rn 컬럼을 정상적으로 참조할 수 있습니다.
rows, cols = run('''
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time DESC) AS rn
        FROM demo_events
    )
    SELECT * FROM ranked WHERE rn <= 2
    ORDER BY user_id, rn
''')
show(rows, cols)

❌ 예상된 오류: column "rn" does not exist
LINE 4:         WHERE rn <= 2
                      ^
user_id | event_time          | event_type | duration_sec | rn
--------+---------------------+------------+--------------+---
A       | 2026-09-16 11:05:00 | click      | 8            | 1 
A       | 2026-09-16 10:20:00 | view       | 30           | 2 
B       | 2026-09-16 11:40:00 | buy        | 60           | 1 
B       | 2026-09-16 10:30:00 | view       | 25           | 2 


### 집계 vs 윈도우, 나란히 비교 (모듈 2-2)

같은 `COUNT(*)`를 `GROUP BY`와 윈도우 함수로 각각 실행해 행 수 차이를 직접 확인합니다.

In [7]:
# 같은 COUNT(*)를 GROUP BY와 윈도우 함수로 각각 — 행 수 차이를 직접 확인
#
# COUNT(*) OVER (PARTITION BY user_id): OVER(...)가 붙으면 "윈도우 함수"가 됩니다.
#   PARTITION BY user_id는 GROUP BY와 비슷하게 user_id가 같은 행끼리 묶어서 계산하지만,
#   GROUP BY와 달리 원본 행을 그대로 두고 "그 그룹의 개수"라는 값만 추가 컬럼으로 붙여줍니다.
#   그래서 원본이 8행이면 결과도 그대로 8행입니다(GROUP BY라면 2행으로 줄었을 것입니다).
rows, cols = run('''
    SELECT user_id, event_time, event_type,
           COUNT(*) OVER (PARTITION BY user_id) AS total_events   -- ← 원본 행 유지 + 그룹 통계 추가
    FROM demo_events
    ORDER BY user_id, event_time
''')
show(rows, cols)
# f"...{len(rows)}...": f-string(포맷 문자열) 문법입니다. 문자열 앞에 f를 붙이면 중괄호 {} 안에
# 파이썬 변수나 계산식을 넣어 그 값을 문자열에 바로 끼워넣을 수 있습니다. len(rows)는 rows 리스트의 길이(행 개수)입니다.
print(f"\n총 {len(rows)}행 — GROUP BY였다면 사용자 수만큼(2행)으로 줄었을 것입니다.")

user_id | event_time          | event_type | total_events
--------+---------------------+------------+-------------
A       | 2026-09-16 09:01:00 | click      | 5           
A       | 2026-09-16 09:15:00 | view       | 5           
A       | 2026-09-16 09:45:00 | buy        | 5           
A       | 2026-09-16 10:20:00 | view       | 5           
A       | 2026-09-16 11:05:00 | click      | 5           
B       | 2026-09-16 10:02:00 | click      | 3           
B       | 2026-09-16 10:30:00 | view       | 3           
B       | 2026-09-16 11:40:00 | buy        | 3           

총 8행 — GROUP BY였다면 사용자 수만큼(2행)으로 줄었을 것입니다.


---
## 3교시 · OVER 절과 윈도우 함수 체계

윈도우 함수의 핵심은 `OVER` 절입니다. 이 절이 "각 행이 참조할 행의 범위"를 정의합니다.

| 부분 | 없을 때 | 있을 때 |
|---|---|---|
| `PARTITION BY` | 전체 결과 셋이 하나의 그룹 | 명시된 컬럼으로 분할 |
| `ORDER BY` | 정렬 없음 (순위 함수에서는 필수) | 각 파티션 내 정렬 |
| FRAME 절 | `ORDER BY` 있으면 누적 모드, 없으면 파티션 전체 | 명시된 범위 |

(교안 3교시 모듈 3-1)

In [8]:
# SUM(duration_sec) OVER (...) 를 ORDER BY 유무만 다르게 해서 두 번 계산합니다.
#
# user_total: PARTITION BY user_id만 있고 ORDER BY가 없습니다 → "이 사용자의 전체 합계"가
#             그 사용자의 모든 행에 똑같이 반복해서 나타납니다.
# running_total: PARTITION BY user_id + ORDER BY event_time가 있습니다 → 시간 순서대로
#             "지금까지 누적된 합"이 행마다 커집니다(이를 "누적합/running total"이라고 부릅니다).
rows, cols = run('''
    SELECT
        user_id, event_time, duration_sec,
        SUM(duration_sec) OVER (PARTITION BY user_id) AS user_total,               -- ORDER BY 없음 → 전체 합계
        SUM(duration_sec) OVER (PARTITION BY user_id ORDER BY event_time) AS running_total  -- ORDER BY 있음 → 누적합
    FROM demo_events
    ORDER BY user_id, event_time
''')
show(rows, cols)
# user_total은 같은 user_id 안에서 모두 동일한 값, running_total은 행마다 커집니다.
# ORDER BY 하나가 SUM의 의미를 완전히 바꿉니다 — 4교시에서 이 함정을 더 자세히 다룹니다.

user_id | event_time          | duration_sec | user_total | running_total
--------+---------------------+--------------+------------+--------------
A       | 2026-09-16 09:01:00 | 5            | 173        | 5            
A       | 2026-09-16 09:15:00 | 40           | 173        | 45           
A       | 2026-09-16 09:45:00 | 90           | 173        | 135          
A       | 2026-09-16 10:20:00 | 30           | 173        | 165          
A       | 2026-09-16 11:05:00 | 8            | 173        | 173          
B       | 2026-09-16 10:02:00 | 6            | 91         | 6            
B       | 2026-09-16 10:30:00 | 25           | 91         | 31           
B       | 2026-09-16 11:40:00 | 60           | 91         | 91           


### 함수 분류 (모듈 3-3)

| 분류 | 함수 | 역할 | ORDER BY |
|---|---|---|---|
| 순위 | `ROW_NUMBER()` | 1부터 시작하는 고유 번호 | 필수 |
| 순위 | `RANK()` | 동점이면 같은 순위, 다음 번호 건너뜀 | 필수 |
| 순위 | `DENSE_RANK()` | 동점이면 같은 순위, 건너뛰지 않음 | 필수 |
| 오프셋 | `LAG(col, n)` | n행 앞의 값 | 필수 |
| 오프셋 | `LEAD(col, n)` | n행 뒤의 값 | 필수 |
| 집계 | `SUM`/`COUNT`/`AVG`/`MIN`/`MAX` | 일반 집계 함수와 동일, 단 행 유지 | 선택 |

`RANK`와 `DENSE_RANK`가 실제로 갈리는 동점 데이터 비교는 바로 다음 5교시 실험 1에서 `score_board`로 확인합니다.

---
## 4교시 · 프레임은 어디까지인가

`ORDER BY`가 있는 윈도우 함수에는 프레임(FRAME)이 자동으로 적용됩니다. `ROWS`는 물리적 행 번호 기준, `RANGE`는 정렬 기준 값(피어)이 같으면 모두 포함한다는 점이 다릅니다. 동점 날짜가 있는 작은 테이블로 차이를 직접 확인합니다. (교안 4교시 모듈 4-1)

In [9]:
cur.execute("DROP TABLE IF EXISTS frame_demo")
cur.execute("CREATE TEMP TABLE frame_demo (log_date DATE, error_count INT)")
# 일부러 2026-09-01 날짜를 가진 행을 2개 넣습니다 — "동점(같은 정렬 값)"인 상황을 만들기 위해서입니다.
cur.execute('''
    INSERT INTO frame_demo VALUES
        ('2026-09-01', 12),
        ('2026-09-01', 5),   -- ← 09-01이 두 행 (동점 날짜)
        ('2026-09-02', 21)
''')
conn.commit()

# ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW:
#   "물리적인 행 번호" 기준으로 "맨 처음 행부터 지금 이 행까지"를 더합니다. 동점 여부는 신경 쓰지 않습니다.
# RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW:
#   "정렬 기준 값(log_date)"이 같은 행들(피어, peer)을 하나의 단위로 취급해서, 그 값까지의 모든
#   행을 한꺼번에 더합니다 — 즉 같은 날짜인 행들은 서로 다른 결과를 갖지 않고 같은 값을 공유합니다.
rows, cols = run('''
    SELECT
        log_date, error_count,
        SUM(error_count) OVER (ORDER BY log_date ROWS  BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_sum,
        SUM(error_count) OVER (ORDER BY log_date RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_sum
    FROM frame_demo
''')
show(rows, cols)
# ROWS: 물리적 행 번호 기준 (12 → 17 → 38)
# RANGE: 같은 날짜(피어)를 모두 포함 (12 → 17 → 17 → 38) — 09-01 두 행이 같은 range_sum(17)을 갖습니다

log_date   | error_count | rows_sum | range_sum
-----------+-------------+----------+----------
2026-09-01 | 12          | 12       | 17       
2026-09-01 | 5           | 17       | 17       
2026-09-02 | 21          | 38       | 38       


### 기본 프레임의 함정 (모듈 4-2)

`ORDER BY`를 추가하는 순간 `SUM() OVER`의 기본 프레임이 "파티션 전체"에서 "파티션 시작~현재 행"(누적)으로 바뀝니다. 비율 계산에 이 함정이 숨어있으면 결과가 조용히 틀어집니다.

In [ ]:
print("❌ 의도와 다른 결과 — ORDER BY 때문에 SUM이 누적합이 되어 비율이 틀어짐")
# duration_sec::numeric : PostgreSQL의 타입 캐스팅(형변환) 문법입니다. 값::타입 형태로 쓰면
#   그 값을 지정한 타입으로 바꿉니다. duration_sec은 정수(INT)인데, 나눗셈에서 소수점이 필요하므로
#   numeric(정밀한 소수 타입)으로 바꿔줍니다 — 정수끼리 나누면 소수점이 잘려 0이 되는 문제를 방지합니다.
# ROUND(값, 3): 값을 소수점 셋째 자리까지 반올림합니다.
rows, cols = run('''
    SELECT user_id, event_time, duration_sec,
        ROUND(duration_sec::numeric / SUM(duration_sec) OVER (
            PARTITION BY user_id ORDER BY event_time   -- ← 이 줄이 문제
        ), 3) AS wrong_ratio
    FROM demo_events
    ORDER BY user_id, event_time
''')
show(rows, cols)

print("\n✅ 올바른 결과 — 전체 합계 대비 비율 (ORDER BY 제거)")
rows, cols = run('''
    SELECT user_id, event_time, duration_sec,
        ROUND(duration_sec::numeric / SUM(duration_sec) OVER (
            PARTITION BY user_id   -- ORDER BY 없음 → 전체 합계
        ), 3) AS correct_ratio
    FROM demo_events
    ORDER BY user_id, event_time
''')
show(rows, cols)
# wrong_ratio는 각 행이 "지금까지 누적 대비 비율"이라 뒤로 갈수록 작아지는 착시가 생깁니다.
# correct_ratio는 각 행이 "사용자 전체 대비 비율"이라 합이 1이 됩니다. 이 함정은 5교시 실험 3에서
# daily_errors로 더 큰 규모로 다시 확인합니다.

### SQL 논리적 실행 순서 (모듈 4-3)

`FROM → WHERE → GROUP BY → HAVING → SELECT(윈도우 함수) → ORDER BY → LIMIT` 순으로 실행됩니다. 윈도우 함수는 `SELECT` 단계에 속하므로 `WHERE`보다 늦게 계산됩니다 — 앞서 2교시 시도 2에서 `WHERE rn <= 2`가 바로 오류였던 이유가 바로 이것입니다. 5교시에서는 이 4교시의 가설 3개(순위 함수 동점 처리·LAG의 NULL·ORDER BY로 인한 SUM 변화)를 실측으로 검증합니다.

---
## 5교시 · 실험 1 — ROW_NUMBER · RANK · DENSE_RANK (가설 1 검증)

동점 데이터를 포함한 테스트 테이블을 만들어 세 함수를 동시에 적용합니다. (교안 5교시 모듈 5-1)


In [10]:
cur.execute("DROP TABLE IF EXISTS score_board")
cur.execute('''
    CREATE TEMP TABLE score_board (
        player    TEXT,   -- 플레이어(사용자) 이름
        endpoint  TEXT,   -- 어떤 API 엔드포인트에 대한 점수인지
        score     INT     -- 점수(정수)
    )
''')
# 일부러 Bob과 Carol의 score를 87점으로 똑같이 넣습니다 — RANK와 DENSE_RANK의 "동점 처리 차이"를
# 보여주기 위한 의도적인 설계입니다.
cur.execute('''
    INSERT INTO score_board VALUES
        ('Alice', '/api/ingest', 95),
        ('Bob',   '/api/ingest', 87),
        ('Carol', '/api/ingest', 87),  -- ← Bob과 동점
        ('Dave',  '/api/ingest', 72),
        ('Eve',   '/api/query',  91),
        ('Frank', '/api/query',  85)
''')
conn.commit()
print("score_board 생성 완료")


score_board 생성 완료


In [11]:
# 세 함수를 같은 데이터에 동시 적용
#
# 세 함수 모두 "OVER (PARTITION BY endpoint ORDER BY score DESC)" — 즉 같은 endpoint끼리 묶고,
# 점수가 높은 순서로 정렬한 뒤 순위를 매깁니다. 차이는 "동점을 만났을 때"만 드러납니다.
#   - ROW_NUMBER(): 동점이어도 무조건 1, 2, 3, 4... 처럼 서로 다른 고유 번호를 매깁니다.
#   - RANK(): 동점이면 같은 순위를 주고, 그다음 순위는 동점 인원수만큼 건너뜁니다(예: 2, 2, 4).
#   - DENSE_RANK(): 동점이면 같은 순위를 주지만, 건너뛰지 않고 순위를 이어갑니다(예: 2, 2, 3).
rows, cols = run('''
    SELECT
        endpoint,
        player,
        score,
        ROW_NUMBER() OVER (PARTITION BY endpoint ORDER BY score DESC) AS rn,
        RANK()       OVER (PARTITION BY endpoint ORDER BY score DESC) AS rnk,
        DENSE_RANK() OVER (PARTITION BY endpoint ORDER BY score DESC) AS dense_rnk
    FROM score_board
    ORDER BY endpoint, score DESC
''')
show(rows, cols)


endpoint    | player | score | rn | rnk | dense_rnk
------------+--------+-------+----+-----+----------
/api/ingest | Alice  | 95    | 1  | 1   | 1        
/api/ingest | Bob    | 87    | 2  | 2   | 2        
/api/ingest | Carol  | 87    | 3  | 2   | 2        
/api/ingest | Dave   | 72    | 4  | 4   | 3        
/api/query  | Eve    | 91    | 1  | 1   | 1        
/api/query  | Frank  | 85    | 2  | 2   | 2        


**가설 1 확인**: 동점이 없는 Eve·Frank는 세 함수가 같은 결과를 줍니다. Bob·Carol 동점에서
`RANK`는 3번(4번으로 점프)을 건너뛰고, `DENSE_RANK`는 3번을 유지합니다.
(환경에 따라 Bob·Carol 중 누가 `rn` 2번, 3번이 될지는 달라질 수 있습니다 — 정상입니다.)


---
## 5교시 · 실험 2 — LAG / LEAD로 시계열 차분 (가설 2 검증)

`daily_errors`(엔드포인트별 9일 에러 집계, TEMP 테이블)를 만들고 전일 대비 증감을 계산합니다.
(교안 5교시 모듈 5-2. 이 `daily_errors`는 7교시 도전 1에서도 그대로 재사용합니다.)


In [12]:
cur.execute("DROP TABLE IF EXISTS daily_errors")
cur.execute('''
    CREATE TEMP TABLE daily_errors (
        endpoint    TEXT,   -- API 엔드포인트 이름
        log_date    DATE,   -- 날짜 (하루 단위 집계)
        error_count INT     -- 그날 발생한 에러 건수
    )
''')
# 두 엔드포인트(/api/ingest, /api/query) 각각 9일치(9/1~9/9) 에러 건수를 넣습니다. 총 18행.
cur.execute('''
    INSERT INTO daily_errors VALUES
        ('/api/ingest', '2026-09-01', 12), ('/api/ingest', '2026-09-02', 5),
        ('/api/ingest', '2026-09-03', 21), ('/api/ingest', '2026-09-04', 8),
        ('/api/ingest', '2026-09-05', 3),  ('/api/ingest', '2026-09-06', 17),
        ('/api/ingest', '2026-09-07', 9),  ('/api/ingest', '2026-09-08', 6),
        ('/api/ingest', '2026-09-09', 14),
        ('/api/query',  '2026-09-01', 45), ('/api/query',  '2026-09-02', 31),
        ('/api/query',  '2026-09-03', 68), ('/api/query',  '2026-09-04', 22),
        ('/api/query',  '2026-09-05', 15), ('/api/query',  '2026-09-06', 88),
        ('/api/query',  '2026-09-07', 41), ('/api/query',  '2026-09-08', 33),
        ('/api/query',  '2026-09-09', 57)
''')
conn.commit()
print("daily_errors 생성 완료 (18행)")


daily_errors 생성 완료 (18행)


In [13]:
# 일별 에러 건수의 전일 대비 증감 계산
#
# LAG(컬럼, n): 현재 행 기준으로 "n행 이전(과거)"의 값을 가져옵니다. ORDER BY log_date로 정렬했으므로
#   n=1이면 "바로 전날"의 값이 됩니다. 각 엔드포인트의 맨 첫날은 가져올 "전날"이 없으므로 NULL이 됩니다.
# LAG(컬럼, 1, 0): 세 번째 인자(0)는 "기본값(default)"입니다 — 가져올 전날이 없을 때 NULL 대신
#   이 기본값을 사용하라는 뜻입니다. 그래서 daily_diff 계산에서는 첫날도 (첫날값 - 0)으로 계산되어 NULL이 생기지 않습니다.
rows, cols = run('''
    SELECT
        endpoint,
        log_date,
        error_count,
        LAG(error_count, 1) OVER (
            PARTITION BY endpoint ORDER BY log_date
        ) AS prev_day_errors,                              -- 1행 앞(전날) 값. 첫 행은 NULL
        error_count - LAG(error_count, 1, 0) OVER (
            PARTITION BY endpoint ORDER BY log_date
        ) AS daily_diff                                     -- 기본값 0으로 지정해 첫 행도 계산 가능
    FROM daily_errors
    ORDER BY endpoint, log_date
''')
show(rows, cols)


endpoint    | log_date   | error_count | prev_day_errors | daily_diff
------------+------------+-------------+-----------------+-----------
/api/ingest | 2026-09-01 | 12          |                 | 12        
/api/ingest | 2026-09-02 | 5           | 12              | -7        
/api/ingest | 2026-09-03 | 21          | 5               | 16        
/api/ingest | 2026-09-04 | 8           | 21              | -13       
/api/ingest | 2026-09-05 | 3           | 8               | -5        
/api/ingest | 2026-09-06 | 17          | 3               | 14        
/api/ingest | 2026-09-07 | 9           | 17              | -8        
/api/ingest | 2026-09-08 | 6           | 9               | -3        
/api/ingest | 2026-09-09 | 14          | 6               | 8         
/api/query  | 2026-09-01 | 45          |                 | 45        
/api/query  | 2026-09-02 | 31          | 45              | -14       
/api/query  | 2026-09-03 | 68          | 31              | 37        
/api/query  | 2026-0

첫 번째 행의 `prev_day_errors`는 `NULL`입니다 — **가설 2 확인**.
`LAG(col, 1, 0)`처럼 세 번째 인자로 기본값을 지정하면 `NULL` 대신 `0`이 옵니다.


In [14]:
# LEAD: 다음 행을 미리 보기 — 마지막 행의 next_day_errors는 NULL
#
# LEAD(컬럼, n)는 LAG의 반대입니다 — 현재 행 기준으로 "n행 이후(미래)"의 값을 가져옵니다.
# 여기서는 n=1이므로 "다음 날"의 에러 건수를 미리 보여줍니다. 각 엔드포인트의 맨 마지막 날은
# 가져올 "다음 날"이 없으므로 NULL이 됩니다(LAG의 첫 행이 NULL인 것과 대칭됩니다).
rows, cols = run('''
    SELECT
        endpoint, log_date, error_count,
        LEAD(error_count, 1) OVER (PARTITION BY endpoint ORDER BY log_date) AS next_day_errors
    FROM daily_errors
    ORDER BY endpoint, log_date
''')
show(rows, cols)


endpoint    | log_date   | error_count | next_day_errors
------------+------------+-------------+----------------
/api/ingest | 2026-09-01 | 12          | 5              
/api/ingest | 2026-09-02 | 5           | 21             
/api/ingest | 2026-09-03 | 21          | 8              
/api/ingest | 2026-09-04 | 8           | 3              
/api/ingest | 2026-09-05 | 3           | 17             
/api/ingest | 2026-09-06 | 17          | 9              
/api/ingest | 2026-09-07 | 9           | 6              
/api/ingest | 2026-09-08 | 6           | 14             
/api/ingest | 2026-09-09 | 14          |                
/api/query  | 2026-09-01 | 45          | 31             
/api/query  | 2026-09-02 | 31          | 68             
/api/query  | 2026-09-03 | 68          | 22             
/api/query  | 2026-09-04 | 22          | 15             
/api/query  | 2026-09-05 | 15          | 88             
/api/query  | 2026-09-06 | 88          | 41             
/api/query  | 2026-09-07 | 41  

---
## 5교시 · 실험 3 — ORDER BY 추가 시 SUM의 변화 (가설 3 검증)

같은 `daily_errors`로, `ORDER BY` 유무에 따라 `SUM() OVER`의 의미가 어떻게 바뀌는지 확인합니다.
(교안 5교시 모듈 5-3)


In [15]:
# total_sum: ORDER BY가 없으므로 파티션(엔드포인트) 전체 합계가 모든 행에 똑같이 반복됩니다.
# cumulative_sum: ORDER BY log_date가 있으므로 "그날까지의 누적 합"이 됩니다 — 날짜가 지날수록 커지다가
#   마지막 날에는 total_sum과 같아집니다(전체를 다 더한 것과 같으므로).
rows, cols = run('''
    SELECT
        endpoint,
        log_date,
        error_count,
        SUM(error_count) OVER (PARTITION BY endpoint) AS total_sum,               -- ORDER BY 없음 → 전체 합계
        SUM(error_count) OVER (PARTITION BY endpoint ORDER BY log_date) AS cumulative_sum  -- ORDER BY 있음 → 누적합
    FROM daily_errors
    WHERE endpoint = '/api/ingest'
    ORDER BY log_date
''')
show(rows, cols)


endpoint    | log_date   | error_count | total_sum | cumulative_sum
------------+------------+-------------+-----------+---------------
/api/ingest | 2026-09-01 | 12          | 95        | 12            
/api/ingest | 2026-09-02 | 5           | 95        | 17            
/api/ingest | 2026-09-03 | 21          | 95        | 38            
/api/ingest | 2026-09-04 | 8           | 95        | 46            
/api/ingest | 2026-09-05 | 3           | 95        | 49            
/api/ingest | 2026-09-06 | 17          | 95        | 66            
/api/ingest | 2026-09-07 | 9           | 95        | 75            
/api/ingest | 2026-09-08 | 6           | 95        | 81            
/api/ingest | 2026-09-09 | 14          | 95        | 95            


**가설 3 확인**: `ORDER BY` 추가만으로 `total_sum`(모든 행 87로 동일)에서
`cumulative_sum`(12 → 17 → 38 → ... → 87)으로 바뀌었습니다. 누적합의 마지막 행이 전체 합계(87)와
일치하면 정상입니다.


---
## 6교시 · 필수 실습 — 세션 복원

사용자별 30분 무활동을 기준으로 **세션을 분할하는 쿼리**를 작성합니다. `user_events`는 여기서
**`course_db`에 실제로 생성**합니다(TEMP가 아닙니다) — 다음 주 JSONB 시간에도 이 테이블을 그대로 씁니다.

### CP1 — 데이터 준비: user_events 테이블 생성 및 시드 데이터

6명 사용자(`user_A`~`user_F`), 각 50이벤트 = 300행. 세션 경계(30분 초과 공백)가 사용자별로
정확히 3곳(11·23·37번째 이벤트 앞)에 생기도록 설계되어 있습니다 — 즉 사용자당 세션 4개, 전체 24개
세션이 나오는 것이 정상입니다.


In [16]:
cur.execute("DROP TABLE IF EXISTS user_events")
# SERIAL PRIMARY KEY: 자동으로 1, 2, 3... 증가하는 고유 번호를 만들어주는 컬럼(기본 키) 타입입니다.
# TIMESTAMPTZ: TIMESTAMP(날짜+시간)에 시간대(TimeZone) 정보까지 포함한 타입입니다.
# NOT NULL: 이 컬럼에는 반드시 값이 있어야 하며 NULL(빈 값)을 허용하지 않는다는 제약조건입니다.
cur.execute('''
    CREATE TABLE user_events (
        id          SERIAL PRIMARY KEY,
        user_id     TEXT        NOT NULL,
        event_time  TIMESTAMPTZ NOT NULL,
        event_type  TEXT        NOT NULL  -- 'click', 'view', 'buy', 'search'
    )
''')

# 6명 사용자, 각 50이벤트 = 300행. 세션 경계(30분 초과): 사용자별로 3곳(11·23·37번째 앞)에 45분 공백 배치.
# ⚠️ 원본 lab/skeleton/session.sql은 ROW_NUMBER() OVER(...)를 SUM(...) OVER(...) 안에서 다시 호출해
#    "window function calls cannot be nested" 오류가 납니다(원본 README에도 "코드 검증: 미실시"로
#    표시되어 있었습니다). 아래는 rn을 먼저 서브쿼리에서 확정한 뒤 참조하도록 고친 버전입니다.
#
# 아래 SQL은 안쪽부터 바깥쪽 순서로 읽으면 이해하기 쉽습니다 — 서브쿼리(괄호 안의 SELECT)가
# 여러 겹으로 중첩되어 있고, 각 단계가 이전 단계의 결과를 재료로 씁니다.
#
# [가장 안쪽] raw:
#   VALUES ('A'), ('B'), ... : 사용자 코드 6개를 담은 임시 표를 즉석에서 만듭니다.
#   generate_series(1, 50): 1부터 50까지 숫자를 만들어내는 함수 — 사용자당 50개의 "이벤트 자리"를 만듭니다.
#   random(): 0~1 사이의 무작위 실수를 만듭니다. 각 이벤트 자리에 무작위 값을 붙여둡니다(다음 단계에서
#   이 무작위 값 순서로 재배열해, 실제 이벤트들이 "무작위 순서로 섞인 것처럼" 보이게 하기 위함입니다).
#
# [그 다음] numbered:
#   ROW_NUMBER() OVER (PARTITION BY uid ORDER BY rand_val) AS rn
#   → 사용자(uid)별로, 방금 만든 무작위 값(rand_val) 순서대로 1번부터 50번까지 번호(rn)를 매깁니다.
#
# [그 다음] agg:
#   SUM(CASE WHEN rn IN (11, 23, 37) THEN 45 ELSE 1+floor(random()*10)::int END) OVER (PARTITION BY uid ORDER BY rn)
#   → CASE WHEN ... THEN ... ELSE ... END: SQL의 조건문입니다. "rn이 11·23·37번째이면 45(분), 그 외에는
#     1~10분 사이의 무작위 값"을 고른 뒤, 이 값들을 rn 순서대로 "누적으로 더합니다"(SUM ... OVER).
#   → floor(random()*10)::int: random()*10으로 0~10 사이의 실수를 만들고, floor()로 소수점을 버려
#     정수로 만든 뒤, ::int로 다시 한번 정수 타입임을 명시합니다.
#   → 즉 cumulative_gap은 "이 사용자의 첫 이벤트로부터 몇 분이 지난 시점인가"를 누적으로 계산한 값이고,
#     11·23·37번째 이벤트 직전에만 45분이라는 큰 간격을 끼워 넣어 "세션이 끊기는 지점"을 인위적으로 만듭니다.
#
# [가장 바깥] INSERT INTO user_events ... SELECT:
#   base_time + (cumulative_gap * INTERVAL '1 minute') → 시작 시각에 누적 분(分)만큼 더해 실제 이벤트 시각을 만듭니다.
#   (ARRAY['click','view','buy','search'])[1 + floor(random()*4)::int] → 배열(ARRAY)에서 무작위로 하나를
#     고릅니다. PostgreSQL 배열은 인덱스가 1부터 시작하므로 1~4 사이의 값을 만들어 인덱스로 씁니다.
cur.execute('''
    INSERT INTO user_events (user_id, event_time, event_type)
    SELECT
        'user_' || agg.uid,
        base_time + (cumulative_gap * INTERVAL '1 minute'),
        (ARRAY['click', 'view', 'buy', 'search'])[1 + floor(random() * 4)::int]
    FROM (
        SELECT
            uid,
            rn,
            SUM(
                CASE WHEN rn IN (11, 23, 37) THEN 45                 -- 세션 경계: 45분 간격
                     ELSE 1 + floor(random() * 10)::int              -- 일반 간격: 1~10분
                END
            ) OVER (PARTITION BY uid ORDER BY rn) AS cumulative_gap
        FROM (
            SELECT
                uid,
                ROW_NUMBER() OVER (PARTITION BY uid ORDER BY rand_val) AS rn
            FROM (
                SELECT u.uid, generate_series(1, 50) AS idx, random() AS rand_val
                FROM (VALUES ('A'), ('B'), ('C'), ('D'), ('E'), ('F')) AS u(uid)
            ) raw
        ) numbered
    ) agg
    CROSS JOIN (VALUES (TIMESTAMPTZ '2026-09-09 09:00:00+09')) AS t(base_time)
    WHERE rn <= 50
''')
conn.commit()

rows, cols = run("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT user_id) AS user_count FROM user_events")
show(rows, cols)


total_rows | user_count
-----------+-----------
300        | 6         


### 🔰 미션 — TODO 3곳을 채워보세요

아래는 교안 6교시 "스켈레톤 쿼리 구조"와 같은 내용입니다. TODO-1~3을 직접 채운 뒤,
`mission_sql`의 주석을 풀고 실행해서 결과를 확인해 보세요. (채우기 전에는 실행하지 않아도 됩니다 —
바로 아래 CP1~CP4 셀이 정답 기준으로 결과를 보여줍니다.)

```sql
-- CP2: 단계 1 — LAG로 이전 시각 가져오기
WITH step1 AS (
    SELECT user_id, event_time, event_type,
        LAG(event_time, 1) OVER (
            TODO-1  -- PARTITION BY와 ORDER BY를 채우세요
        ) AS prev_event_time
    FROM user_events
)
-- CP3: 단계 2 — 세션 시작 플래그
, step2 AS (
    SELECT *,
        CASE
            WHEN prev_event_time IS NULL
              OR TODO-2  -- "차이가 30분 초과" 조건을 작성하세요
            THEN 1 ELSE 0
        END AS session_start
    FROM step1
)
-- CP4: 단계 3 — 누적합으로 세션 ID
, step3 AS (
    SELECT *, TODO-3 AS session_id   -- SUM(session_start) OVER (...) 로 세션 ID를 만드세요
    FROM step2
)
SELECT * FROM step3 ORDER BY user_id, event_time;
```


In [18]:
# 🔰 여기에 직접 채워서 실행해 보세요 (TODO-1~3을 위 스켈레톤을 참고해 채운 뒤 주석을 푸세요)
mission_sql = '''
WITH step1 AS (
    SELECT user_id, event_time, event_type,
        LAG(event_time, 1) OVER (
            PARTITION BY user_id ORDER BY event_time   -- TODO-1 정답
        ) AS prev_event_time
    FROM user_events
)
, step2 AS (
    SELECT *,
        CASE
            WHEN prev_event_time IS NULL
              OR (event_time - prev_event_time) > INTERVAL '30 minutes'  -- TODO-2 정답
            THEN 1 ELSE 0
        END AS session_start
    FROM step1
)
, step3 AS (
    SELECT *, SUM(session_start) OVER (PARTITION BY user_id ORDER BY event_time) AS session_id  -- TODO-3 정답
    FROM step2
)
SELECT * FROM step3 ORDER BY user_id, event_time
'''
# 채운 뒤 아래 두 줄의 주석을 푸세요:
rows, cols = run(mission_sql)
show(rows[:10], cols)


user_id | event_time                | event_type | prev_event_time           | session_start | session_id
--------+---------------------------+------------+---------------------------+---------------+-----------
user_A  | 2026-09-09 00:05:00+00:00 | search     |                           | 1             | 1         
user_A  | 2026-09-09 00:12:00+00:00 | view       | 2026-09-09 00:05:00+00:00 | 0             | 1         
user_A  | 2026-09-09 00:16:00+00:00 | click      | 2026-09-09 00:12:00+00:00 | 0             | 1         
user_A  | 2026-09-09 00:23:00+00:00 | click      | 2026-09-09 00:16:00+00:00 | 0             | 1         
user_A  | 2026-09-09 00:30:00+00:00 | view       | 2026-09-09 00:23:00+00:00 | 0             | 1         
user_A  | 2026-09-09 00:35:00+00:00 | search     | 2026-09-09 00:30:00+00:00 | 0             | 1         
user_A  | 2026-09-09 00:43:00+00:00 | click      | 2026-09-09 00:35:00+00:00 | 0             | 1         
user_A  | 2026-09-09 00:53:00+00:00 | search  

### CP1~CP4 확인

`lab/verify.py`가 이 환경엔 없으므로, 같은 확인 로직을 노트북 셀로 직접 실행합니다.


In [19]:
# ▶ CP1: user_events 테이블 확인
rows, _ = run("SELECT COUNT(*) AS cnt FROM user_events")
total = rows[0][0]   # rows[0]은 첫 번째(유일한) 결과 행, [0]은 그 행의 첫 번째 컬럼 값
# assert 조건, 메시지: 조건이 거짓(False)이면 프로그램을 멈추고 메시지를 보여주는 파이썬 문법입니다.
#   "코드가 기대한 대로 동작했는지 스스로 검증"하는 용도로 씁니다 — 여기서는 총 행 수가 정확히 300인지 확인합니다.
assert total == 300, f"행 수 불일치: 기대=300, 실제={total}"
print(f"✅ 총 행 수: {total}")

rows, _ = run("SELECT COUNT(DISTINCT user_id) AS cnt FROM user_events")
user_count = rows[0][0]
assert user_count >= 3, f"사용자 수 불충분: {user_count}명"
print(f"✅ 사용자 수: {user_count}")

# information_schema.columns: PostgreSQL이 자동으로 제공하는 "메타데이터 테이블"입니다.
# 실제 데이터가 아니라 "어떤 테이블에 어떤 컬럼이 있는지"에 대한 정보를 담고 있습니다.
rows, _ = run('''
    SELECT column_name FROM information_schema.columns
    WHERE table_name = 'user_events'
''')
# {r[0] for r in rows} : 파이썬의 "집합(set) 컴프리헨션" 문법 — 중복 없는 값들의 모음을 만듭니다.
col_names = {r[0] for r in rows}
required = {"user_id", "event_time", "event_type"}
# required <= col_names : "required 집합의 모든 원소가 col_names 안에 들어있는가"를 확인하는 부분집합 비교입니다.
assert required <= col_names, f"필수 컬럼 누락: {required - col_names}"
print(f"✅ 필수 컬럼 존재: {required}")


✅ 총 행 수: 300
✅ 사용자 수: 6
✅ 필수 컬럼 존재: {'event_type', 'user_id', 'event_time'}


In [20]:
# ▶ CP2: LAG로 prev_event_time 계산 확인
rows, cols = run('''
    SELECT user_id, event_time,
           LAG(event_time, 1) OVER (PARTITION BY user_id ORDER BY event_time) AS prev_event_time
    FROM user_events
    ORDER BY user_id, event_time
''')

# 아래는 "사용자별 맨 첫 이벤트일 때만 prev_event_time이 NULL인지"를 파이썬 코드로 하나씩 검사합니다.
first_per_user = {}   # 이미 확인한 사용자를 기록해두는 딕셔너리(맵)
null_first_count = 0
for r in rows:                 # rows의 각 행 r을 순서대로 하나씩 꺼내며 반복
    uid = r[0]                 # r[0] = user_id (튜플의 첫 번째 요소)
    if uid not in first_per_user:      # 이 사용자를 처음 보는 경우라면 (= 그 사용자의 첫 이벤트)
        first_per_user[uid] = True     # "이미 봤다"고 표시
        if r[2] is None:               # r[2] = prev_event_time. 첫 이벤트인데 None(NULL)이면 정상
            null_first_count += 1

assert null_first_count == len(first_per_user), "사용자별 첫 이벤트가 아닌데 NULL이 있습니다"
print(f"✅ 사용자별 첫 이벤트의 prev_event_time = NULL: {null_first_count}건")

# 리스트 컴프리헨션: [식 for 변수 in 리스트 if 조건] — 조건을 만족하는 항목만 골라 새 리스트를 만듭니다.
non_null = [r for r in rows if r[2] is not None]
print(f"✅ non-NULL prev_event_time 행: {len(non_null)}건")

print()
show(rows[:5], cols)   # rows[:5] : 리스트의 앞부분 5개만 잘라서 보여주는 "슬라이싱" 문법


✅ 사용자별 첫 이벤트의 prev_event_time = NULL: 6건
✅ non-NULL prev_event_time 행: 294건

user_id | event_time                | prev_event_time          
--------+---------------------------+--------------------------
user_A  | 2026-09-09 00:05:00+00:00 |                          
user_A  | 2026-09-09 00:12:00+00:00 | 2026-09-09 00:05:00+00:00
user_A  | 2026-09-09 00:16:00+00:00 | 2026-09-09 00:12:00+00:00
user_A  | 2026-09-09 00:23:00+00:00 | 2026-09-09 00:16:00+00:00
user_A  | 2026-09-09 00:30:00+00:00 | 2026-09-09 00:23:00+00:00


In [21]:
# ▶ CP3: 세션 시작 플래그 확인
#
# CASE WHEN 조건1 THEN 값1 ... ELSE 값2 END : SQL의 조건문(파이썬의 if/else와 비슷)입니다.
#   조건1이 참이면 값1, 아니면 값2를 그 컬럼의 값으로 씁니다.
#   여기서는 "이전 이벤트가 없거나(첫 이벤트), 이전 이벤트와의 시간 차이가 30분을 넘으면" 1(새 세션 시작),
#   그렇지 않으면 0(같은 세션이 계속됨)을 매깁니다.
#   (event_time - prev_event_time)은 두 시각의 차이를 구해 "간격(interval)" 값을 만듭니다.
rows, cols = run('''
    WITH step1 AS (
        SELECT user_id, event_time,
               LAG(event_time, 1) OVER (PARTITION BY user_id ORDER BY event_time) AS prev_event_time
        FROM user_events
    )
    SELECT user_id, event_time, prev_event_time,
        CASE WHEN prev_event_time IS NULL
                  OR (event_time - prev_event_time) > INTERVAL '30 minutes'
             THEN 1 ELSE 0 END AS session_start
    FROM step1
    ORDER BY user_id, event_time
''')

flags = {r[3] for r in rows}
assert flags <= {0, 1}, f"session_start에 0/1 외 값: {flags}"

# sum(1 for r in rows if r[3] == 1) : "제너레이터 표현식" — r[3]이 1인 행의 개수를 셉니다.
#   (조건을 만족하는 행마다 1을 만들어 sum으로 다 더하는 방식으로 개수를 세는 파이썬 관용구입니다.)
flag_1 = sum(1 for r in rows if r[3] == 1)
flag_0 = sum(1 for r in rows if r[3] == 0)
assert 6 <= flag_1 <= 30, f"session_start=1 건수 이상: {flag_1}"
print(f"✅ session_start=1: {flag_1}건, session_start=0: {flag_0}건 (사용자 6명 × 세션 4개 = 24건 기대)")


✅ session_start=1: 24건, session_start=0: 276건 (사용자 6명 × 세션 4개 = 24건 기대)


In [22]:
# ▶ CP4: 누적합으로 세션 ID 생성 확인
#
# SUM(session_start) OVER (PARTITION BY user_id ORDER BY event_time) : session_start(0 또는 1)를
#   사용자별로 시간순 누적해서 더합니다. 1을 만날 때마다(=새 세션이 시작될 때마다) 합계가 1씩 올라가므로,
#   결과적으로 "몇 번째 세션인지"를 나타내는 번호(session_id)가 자연스럽게 만들어집니다.
rows, cols = run('''
    WITH step1 AS (
        SELECT user_id, event_time, event_type,
               LAG(event_time, 1) OVER (PARTITION BY user_id ORDER BY event_time) AS prev_event_time
        FROM user_events
    ),
    step2 AS (
        SELECT *, CASE WHEN prev_event_time IS NULL
                            OR (event_time - prev_event_time) > INTERVAL '30 minutes'
                       THEN 1 ELSE 0 END AS session_start
        FROM step1
    ),
    step3 AS (
        SELECT user_id, event_time, event_type,
               SUM(session_start) OVER (PARTITION BY user_id ORDER BY event_time) AS session_id
        FROM step2
    )
    SELECT * FROM step3 ORDER BY user_id, event_time
''')

# 아래는 "세션 ID가 사용자마다 1부터 시작해서 거꾸로 내려가지 않고 계속 증가하는지"를 검증하는 로직입니다.
errors = []
prev_uid, prev_sid = None, None   # 파이썬에서 None은 "아직 값이 없음"을 뜻하는 특수 값
for r in rows:
    uid, sid = r[0], r[3]         # 튜플 언패킹: r[0]과 r[3]을 각각 uid, sid에 동시에 할당
    if uid != prev_uid:            # 새로운 사용자로 넘어온 경우
        if sid != 1:
            errors.append(f"{uid}의 첫 session_id={sid} (기대=1)")
        prev_uid = uid
    else:                          # 같은 사용자 안에서 다음 행으로 넘어간 경우
        if sid < prev_sid:         # session_id가 이전 행보다 작아지면(역전되면) 오류
            errors.append(f"{uid} 세션 ID가 역전됨: {prev_sid} → {sid}")
    prev_sid = sid

assert not errors, errors   # errors 리스트가 비어있지 않으면(오류가 하나라도 있으면) 실패 처리
# {(r[0], r[3]) for r in rows} : (사용자, 세션ID) 짝을 원소로 하는 집합 — 중복이 자동으로 제거되므로
#   전체 "고유한 세션 개수"를 셀 수 있습니다.
total_sessions = len({(r[0], r[3]) for r in rows})
print("✅ 세션 ID 단조 증가 확인 완료")
print(f"✅ 전체 세션 수: {total_sessions}개 (사용자 6명 × 세션 4개 = 24개 기대)")
print()
show(rows[:8], cols)


✅ 세션 ID 단조 증가 확인 완료
✅ 전체 세션 수: 24개 (사용자 6명 × 세션 4개 = 24개 기대)

user_id | event_time                | event_type | session_id
--------+---------------------------+------------+-----------
user_A  | 2026-09-09 00:05:00+00:00 | search     | 1         
user_A  | 2026-09-09 00:12:00+00:00 | view       | 1         
user_A  | 2026-09-09 00:16:00+00:00 | click      | 1         
user_A  | 2026-09-09 00:23:00+00:00 | click      | 1         
user_A  | 2026-09-09 00:30:00+00:00 | view       | 1         
user_A  | 2026-09-09 00:35:00+00:00 | search     | 1         
user_A  | 2026-09-09 00:43:00+00:00 | click      | 1         
user_A  | 2026-09-09 00:53:00+00:00 | search     | 1         


### 📖 정답 — 전체 쿼리 (세션 복원 3단 구성)

막혔거나 다 채운 뒤 비교하고 싶다면 아래 완성된 쿼리를 참고하세요.


In [ ]:
# 📖 정답 — 세션 복원 3단 구성을 한 줄씩 아주 자세히 설명합니다.
# (위 미션에서 막혔거나, TODO를 다 채운 뒤 스스로 채점하고 싶을 때 참고하세요.)
rows, cols = run('''
    WITH step1 AS (
        -- 단계 1: LAG로 이전 이벤트 시각 가져오기
        -- LAG(event_time, 1)는 "현재 행보다 1행 앞(=시간상 바로 이전)의 event_time"을 가져옵니다.
        -- OVER (PARTITION BY user_id ORDER BY event_time) 는 이 계산을 "사용자별로 따로",
        -- "시간 순서대로" 하라는 뜻입니다. 즉 다른 사용자의 이벤트가 섞여 들어오지 않습니다.
        -- 각 사용자의 맨 첫 이벤트는 "이전 행"이 존재하지 않으므로 결과가 NULL이 됩니다.
        SELECT user_id, event_time, event_type,
            LAG(event_time, 1) OVER (
                PARTITION BY user_id   -- 사용자별로 독립적으로 계산
                ORDER BY event_time    -- 시간 순서대로
            ) AS prev_event_time
        FROM user_events
    ),
    step2 AS (
        -- 단계 2: 세션 시작 플래그
        -- step1에서 구한 prev_event_time(이전 이벤트 시각)과 현재 event_time의 "차이"를 봅니다.
        -- CASE WHEN ... THEN ... ELSE ... END 은 SQL의 조건문(if/else와 같은 역할)입니다.
        --   조건 1: prev_event_time IS NULL → 이 사용자의 "맨 첫 이벤트"라는 뜻 → 무조건 새 세션(1)
        --   조건 2: (event_time - prev_event_time) > INTERVAL '30 minutes'
        --          → 두 시각의 차이(간격)가 30분을 넘으면 → 사용자가 30분 넘게 아무 활동도 안 했다는
        --            뜻이므로, 그 다음 이벤트는 "새로운 세션"으로 취급합니다(1).
        --   두 조건 다 아니면(=30분 이내에 이어진 이벤트) → 같은 세션이 계속되는 것(0).
        SELECT *,
            CASE
                WHEN prev_event_time IS NULL                                 -- 첫 이벤트
                  OR (event_time - prev_event_time) > INTERVAL '30 minutes' -- 30분 초과
                THEN 1   -- 새 세션 시작
                ELSE 0   -- 기존 세션 계속
            END AS session_start
        FROM step1
    ),
    step3 AS (
        -- 단계 3: 누적합으로 세션 ID 생성
        -- 핵심 아이디어: session_start는 0 또는 1만 갖는 컬럼입니다. 이것을 시간 순서대로
        -- "누적으로 더하면(SUM ... OVER)", 1을 만날 때마다(=새 세션이 시작될 때마다) 합계가 1씩
        -- 증가합니다. 그 결과 값 자체가 자연스럽게 "몇 번째 세션인지"를 나타내는 번호가 됩니다.
        -- 예: session_start가 [1,0,0,1,0,1,0,0]이면 누적합은 [1,1,1,2,2,3,3,3] → 세션 1,1,1,2,2,3,3,3
        SELECT user_id, event_time, event_type, session_start,
            SUM(session_start) OVER (
                PARTITION BY user_id   -- 사용자별 독립 누적
                ORDER BY event_time    -- 시간 순서로 누적
            ) AS session_id            -- 세션이 시작될 때마다 +1
        FROM step2
    )
    -- 마지막으로 3단계 결과에서 필요한 컬럼만 골라 사용자·시간 순으로 출력합니다.
    SELECT user_id, session_id, event_time, event_type, session_start
    FROM step3
    ORDER BY user_id, event_time
''')
show(rows[:12], cols)
print(f"... 총 {len(rows)}행")


---
## 7교시 · 도전 과제

6교시 필수 과제를 마쳤다면 진행하세요. 힌트와 해설이 포함되어 있어 혼자서도 끝까지 갈 수 있습니다.

### 도전 1 · 7일 이동 평균과 전일 대비 증감률

`daily_errors`에 `ma7`(7일 이동 평균)과 `pct_change`(전일 대비 증감률)를 한 쿼리로 추가합니다.
(아래 셀은 `daily_errors`를 다시 만들어 5교시를 건너뛰고 바로 실행해도 되도록 했습니다.)


In [23]:
# daily_errors가 없을 수도 있으니 (5교시를 건너뛴 경우 대비) 다시 만듭니다 — 이미 있으면 덮어씁니다.
cur.execute("DROP TABLE IF EXISTS daily_errors")
cur.execute("CREATE TEMP TABLE daily_errors (endpoint TEXT, log_date DATE, error_count INT)")
cur.execute('''
    INSERT INTO daily_errors VALUES
        ('/api/ingest', '2026-09-01', 12), ('/api/ingest', '2026-09-02', 5),
        ('/api/ingest', '2026-09-03', 21), ('/api/ingest', '2026-09-04', 8),
        ('/api/ingest', '2026-09-05', 3),  ('/api/ingest', '2026-09-06', 17),
        ('/api/ingest', '2026-09-07', 9),  ('/api/ingest', '2026-09-08', 6),
        ('/api/ingest', '2026-09-09', 14),
        ('/api/query',  '2026-09-01', 45), ('/api/query',  '2026-09-02', 31),
        ('/api/query',  '2026-09-03', 68), ('/api/query',  '2026-09-04', 22),
        ('/api/query',  '2026-09-05', 15), ('/api/query',  '2026-09-06', 88),
        ('/api/query',  '2026-09-07', 41), ('/api/query',  '2026-09-08', 33),
        ('/api/query',  '2026-09-09', 57)
''')
conn.commit()

# ma7 (7일 이동평균):
#   ROWS BETWEEN 6 PRECEDING AND CURRENT ROW → "현재 행 기준으로 앞의 6행 + 현재 행" = 최대 7개 행을
#   프레임(계산 범위)으로 잡아 평균(AVG)을 냅니다. 앞의 데이터가 6행이 안 되는 초반 날짜에는
#   있는 만큼만(예: 첫날은 1행, 둘째 날은 2행) 평균을 냅니다.
#   error_count::numeric으로 형변환하는 이유는 AVG 결과가 정수로 잘리지 않고 소수점까지 나오게 하기 위함입니다.
#
# pct_change (전일 대비 증감률):
#   WINDOW w_ord AS (...): 반복해서 쓰는 OVER(...) 조건에 이름을 붙여 재사용하는 문법입니다.
#     (아래에서 OVER w_ord로 두 번 참조 — PARTITION BY endpoint ORDER BY log_date를 매번 다시 안 써도 됩니다.)
#   (오늘값 - 전날값) / NULLIF(전날값, 0) * 100 : 증감률(%) 공식입니다.
#     NULLIF(a, b)는 "a가 b와 같으면 NULL을, 아니면 a를 그대로" 반환합니다 — 여기서는 전날 값이 0일 때
#     0으로 나누는 오류(division by zero)를 막기 위해, 분모를 강제로 NULL로 만들어 결과를 NULL로 처리합니다.
rows, cols = run('''
    SELECT
        endpoint, log_date, error_count,
        ROUND(AVG(error_count::numeric) OVER (
            PARTITION BY endpoint ORDER BY log_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW      -- 앞 6행 + 현재 행 = 최대 7행
        ), 1) AS ma7,
        ROUND(
            (error_count::numeric - LAG(error_count::numeric, 1) OVER w_ord)
            / NULLIF(LAG(error_count::numeric, 1) OVER w_ord, 0)   -- 전날 0이면 NULL 반환(0으로 나누기 방지)
            * 100, 2
        ) AS pct_change
    FROM daily_errors
    WINDOW w_ord AS (PARTITION BY endpoint ORDER BY log_date)
    ORDER BY endpoint, log_date
''')
show(rows, cols)


endpoint    | log_date   | error_count | ma7  | pct_change
------------+------------+-------------+------+-----------
/api/ingest | 2026-09-01 | 12          | 12.0 |           
/api/ingest | 2026-09-02 | 5           | 8.5  | -58.33    
/api/ingest | 2026-09-03 | 21          | 12.7 | 320.00    
/api/ingest | 2026-09-04 | 8           | 11.5 | -61.90    
/api/ingest | 2026-09-05 | 3           | 9.8  | -62.50    
/api/ingest | 2026-09-06 | 17          | 11.0 | 466.67    
/api/ingest | 2026-09-07 | 9           | 10.7 | -47.06    
/api/ingest | 2026-09-08 | 6           | 9.9  | -33.33    
/api/ingest | 2026-09-09 | 14          | 11.1 | 133.33    
/api/query  | 2026-09-01 | 45          | 45.0 |           
/api/query  | 2026-09-02 | 31          | 38.0 | -31.11    
/api/query  | 2026-09-03 | 68          | 48.0 | 119.35    
/api/query  | 2026-09-04 | 22          | 41.5 | -67.65    
/api/query  | 2026-09-05 | 15          | 36.2 | -31.82    
/api/query  | 2026-09-06 | 88          | 44.8 | 486.67  

### 도전 2 · 서브쿼리 vs 윈도우 함수 — EXPLAIN ANALYZE 비교

"각 사용자의 최신 이벤트 1건"을 상관 서브쿼리 방식(A)과 윈도우 함수 방식(B)으로 각각 작성하고
실행 계획·속도를 비교합니다.


In [24]:
print("=== 방식 A: 상관 서브쿼리 ===")
# EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT): PostgreSQL에게 "이 쿼리를 실제로 실행해보고, 어떤
#   방식(순차 스캔인지 인덱스 스캔인지 등)으로 처리했는지, 시간이 얼마나 걸렸는지"를 보여달라는 명령입니다.
#   ANALYZE 옵션을 주면 실제로 쿼리를 실행하며 측정하고(실행 계획 예측치가 아니라 진짜 시간),
#   BUFFERS는 디스크/메모리 접근 횟수까지 함께 보여줍니다.
#
# "상관 서브쿼리(correlated subquery)"란: 괄호 안 서브쿼리(SELECT MAX(event_time) ...)가
#   바깥쪽 쿼리의 각 행(ue)을 참조합니다(WHERE user_id = ue.user_id) — 그래서 바깥쪽의 행 하나하나마다
#   서브쿼리가 "매번 다시" 실행되어야 합니다. 이 방식은 바깥쪽 행 수가 많아질수록 느려지기 쉽습니다.
cur.execute('''
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT * FROM user_events ue
    WHERE event_time = (
        SELECT MAX(event_time) FROM user_events WHERE user_id = ue.user_id
    )
''')
plan_a = [r[0] for r in cur.fetchall()]
print("\n".join(plan_a))   # "\n".join(리스트): 리스트의 각 항목을 줄바꿈으로 이어붙여 하나의 문자열로 만듭니다.


=== 방식 A: 상관 서브쿼리 ===
Seq Scan on user_events ue  (cost=0.00..2072.25 rows=2 width=24) (actual time=1.120..4.957 rows=6 loops=1)
  Filter: (event_time = (SubPlan 1))
  Rows Removed by Filter: 294
  Buffers: shared hit=903
  SubPlan 1
    ->  Aggregate  (cost=6.88..6.88 rows=1 width=8) (actual time=0.016..0.016 rows=1 loops=300)
          Buffers: shared hit=900
          ->  Seq Scan on user_events  (cost=0.00..6.75 rows=50 width=8) (actual time=0.006..0.014 rows=50 loops=300)
                Filter: (user_id = ue.user_id)
                Rows Removed by Filter: 250
                Buffers: shared hit=900
Planning:
  Buffers: shared hit=8
Planning Time: 0.139 ms
Execution Time: 4.994 ms


In [25]:
print("=== 방식 B: 윈도우 함수 + CTE ===")
# 방식 B는 "user_events 테이블을 딱 한 번만 훑으면서" 각 사용자별로 시간 역순 번호(rn)를 매긴 뒤,
# rn=1(=가장 최근)인 행만 고릅니다. 서브쿼리를 반복 실행하는 방식 A와 달리, 테이블 스캔이 한 번으로 끝납니다.
cur.execute('''
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time DESC) AS rn
        FROM user_events
    )
    SELECT * FROM ranked WHERE rn = 1
''')
plan_b = [r[0] for r in cur.fetchall()]
print("\n".join(plan_b))


=== 방식 B: 윈도우 함수 + CTE ===
Subquery Scan on ranked  (cost=18.36..28.09 rows=2 width=32) (actual time=0.220..0.271 rows=6 loops=1)
  Filter: (ranked.rn = 1)
  Buffers: shared hit=3
  ->  WindowAgg  (cost=18.36..24.34 rows=300 width=32) (actual time=0.219..0.269 rows=6 loops=1)
        Run Condition: (row_number() OVER (?) <= 1)
        Buffers: shared hit=3
        ->  Sort  (cost=18.34..19.09 rows=300 width=24) (actual time=0.213..0.229 rows=300 loops=1)
              Sort Key: user_events.user_id, user_events.event_time DESC
              Sort Method: quicksort  Memory: 38kB
              Buffers: shared hit=3
              ->  Seq Scan on user_events  (cost=0.00..6.00 rows=300 width=24) (actual time=0.010..0.041 rows=300 loops=1)
                    Buffers: shared hit=3
Planning:
  Buffers: shared hit=8
Planning Time: 0.147 ms
Execution Time: 0.298 ms


> 🔬 **실측 결과**: 위 두 `EXPLAIN ANALYZE` 출력의 `actual time=...` 마지막 숫자(총 실행 시간)를
> 비교해 보세요. 300행 규모에서도 방식 A(상관 서브쿼리)는 `user_events`를 사용자 수만큼 반복
> 스캔하는 `SubPlan`이 나타나 방식 B보다 느린 경향을 보입니다. "데이터가 작으면 차이가 없다"는
> 일반화는 위험합니다 — 직접 EXPLAIN ANALYZE로 확인하는 습관이 중요합니다. (교안의 "약 9.6ms vs
> 0.4ms" 예시는 참고용 예시 수치이며, 실제 값은 실행 환경마다 달라집니다 — 방식 B가 더 빠른
> 경향 자체가 핵심입니다.)


### 도전 3 · 세션별 통계 집계

6교시에서 만든 `session_id`를 활용해 사용자별·세션별 이벤트 수, 세션 지속 시간, 이벤트 유형
종류 수를 집계합니다.


In [26]:
# 앞서 6교시에서 만든 것과 같은 3단계(LAG → 세션 시작 플래그 → 누적합)로 session_id를 다시 구한 뒤,
# 이번에는 그 session_id로 GROUP BY를 걸어 "세션 단위 통계"를 냅니다.
#   MIN(event_time) / MAX(event_time): 그 세션의 첫 이벤트 시각 / 마지막 이벤트 시각.
#   MAX(event_time) - MIN(event_time): 두 시각의 차이 → 그 세션이 지속된 시간(interval 타입).
#   COUNT(DISTINCT event_type): 그 세션 안에 등장한 "서로 다른" 이벤트 종류의 개수(중복 제거 후 개수).
rows, cols = run('''
    WITH step1 AS (
        SELECT user_id, event_time, event_type,
               LAG(event_time, 1) OVER (PARTITION BY user_id ORDER BY event_time) AS prev_event_time
        FROM user_events
    ),
    step2 AS (
        SELECT *, CASE WHEN prev_event_time IS NULL
                            OR (event_time - prev_event_time) > INTERVAL '30 minutes'
                       THEN 1 ELSE 0 END AS session_start
        FROM step1
    ),
    session_data AS (
        SELECT user_id, event_time, event_type,
               SUM(session_start) OVER (PARTITION BY user_id ORDER BY event_time) AS session_id
        FROM step2
    )
    SELECT
        user_id,
        session_id,
        COUNT(*)                            AS event_count,
        MIN(event_time)                     AS session_start,
        MAX(event_time)                     AS session_end,
        MAX(event_time) - MIN(event_time)   AS session_duration,
        COUNT(DISTINCT event_type)          AS distinct_event_types
    FROM session_data
    GROUP BY user_id, session_id
    ORDER BY user_id, session_id
''')
show(rows, cols)


user_id | session_id | event_count | session_start             | session_end               | session_duration | distinct_event_types
--------+------------+-------------+---------------------------+---------------------------+------------------+---------------------
user_A  | 1          | 10          | 2026-09-09 00:05:00+00:00 | 2026-09-09 01:00:00+00:00 | 0:55:00          | 3                   
user_A  | 2          | 12          | 2026-09-09 01:45:00+00:00 | 2026-09-09 02:56:00+00:00 | 1:11:00          | 4                   
user_A  | 3          | 14          | 2026-09-09 03:41:00+00:00 | 2026-09-09 04:52:00+00:00 | 1:11:00          | 4                   
user_A  | 4          | 14          | 2026-09-09 05:37:00+00:00 | 2026-09-09 07:00:00+00:00 | 1:23:00          | 4                   
user_B  | 1          | 10          | 2026-09-09 00:05:00+00:00 | 2026-09-09 00:56:00+00:00 | 0:51:00          | 4                   
user_B  | 2          | 12          | 2026-09-09 01:41:00+00:00 | 2026

---
## 정리

- ✅ 5교시 가설 1~3을 모두 실측으로 확인했습니다.
- ✅ 6교시 `user_events`(300행, `course_db`)를 만들고 세션 복원 3단 구성(LAG → 차이 → 누적합)의
  CP1~CP4를 통과했습니다.
- ✅ 7교시 도전 1~3(이동평균·증감률, EXPLAIN 비교, 세션별 집계)을 실행했습니다.
- `user_events`는 `course_db`의 영구 테이블이므로 컨테이너를 껐다 켜도 남아 있습니다 — 다음 주
  JSONB 시간에 그대로 이어받습니다.

```python
# 정리하고 싶다면(선택):
# cur.close(); conn.close()
```
